# Clase 07: De los documentos al índice - Report Generator
**Laboratorio en Google Colab - Corte 2 - Equipo Juliofi**

En esta práctica prepararemos dos documentos del proyecto, compararemos dos formas de fragmentarlos y guardaremos un índice para volver a consultarlo. Trabajaremos en Colab, con el mismo modelo de embeddings de la clase anterior. La primera ejecución necesita Internet para instalar la biblioteca y descargar el modelo; podemos usar CPU.

Ejecuten las celdas en orden. La consulta se escribe en el punto **5**.

## 1. Preparación
Instalamos la biblioteca y cargamos el modelo. Comprobamos su límite de entrada para evitar que el texto se recorte al generar los embeddings.

In [21]:
%pip -q install sentence-transformers==6.1.0

In [22]:
from pathlib import Path
import re, json, hashlib, zipfile
import numpy as np
from sentence_transformers import SentenceTransformer
from huggingface_hub import model_info

MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
REVISION = model_info(MODELO).sha
modelo = SentenceTransformer(MODELO, revision=REVISION)
tokenizador = modelo.tokenizer
MAX_TOKENS = 128       # Presupuesto del fragmento completo, incluido el encabezado.
SOLAPAMIENTO = 12    # Tokens repetidos entre ventanas de un mismo bloque.

def contar_tokens(texto):
    return len(tokenizador.encode(texto, add_special_tokens=True, truncation=False))

print("Límite del modelo:", modelo.max_seq_length, "tokens")
print("Presupuesto de la práctica:", MAX_TOKENS, "tokens")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Límite del modelo: 128 tokens
Presupuesto de la práctica: 128 tokens


## 2. Los documentos del corpus
Esta celda crea dos archivos Markdown en la sesión. Son una guía de tipos de informe y las reglas de redacción de borradores de Report Generator. Después los leeremos desde la carpeta. Hemos dejado algunos espacios y líneas sobrantes para observar la limpieza.

In [23]:
carpeta = Path("corpus_clase07")
carpeta.mkdir(exist_ok=True)
ORIGEN = "Equipo Juliofi; redactado a partir de SYSTEM_PROMPT_V2 y las reglas de la Clase 04"
fuentes = {
    "tipos_de_informe.md": """# Guía de tipos de informe

## Informe ejecutivo

El informe ejecutivo está dirigido a la dirección y resume los indicadores clave del negocio en un periodo cerrado.  Su estructura tiene cuatro partes: contexto y periodo, indicadores principales, conclusiones y una sección de datos ausentes y riesgos. Cada indicador debe citar el código de la fuente de la que proviene.

Si el periodo solicitado todavía no ha cerrado, el borrador se redacta igual, pero la sección de contexto debe advertir que las cifras son parciales.


## Informe técnico

El informe técnico describe el comportamiento del sistema durante un periodo observado. Las métricas habituales son el uso de CPU, la latencia p95, la disponibilidad y los errores 5xx. El informe presenta las métricas tal como las entrega la fuente; el prototipo no recalcula promedios ni percentiles.
""",
    "reglas_de_borrador.md": """# Reglas de redacción de borradores

## Datos ausentes e incompletos

Cada métrica solicitada se contrasta con las métricas disponibles en la fuente antes de redactar. Una métrica que la fuente no ofrece se escribe como [DATO AUSENTE] en el lugar donde iría la cifra. El borrador no reemplaza ese vacío con una estimación ni con el valor de otro periodo. Si la fuente ofrece la métrica pero con registros faltantes, por ejemplo días sin datos dentro del periodo, la métrica se marca como [DATO INCOMPLETO]. En ese caso la cifra puede aparecer, pero no se presenta como verificada y se explica qué parte del periodo falta.

## Catálogo de fuentes

Las fuentes de datos se identifican con un código de la forma DS-0000, por ejemplo DS-1001. El periodo cubierto, las métricas disponibles y la fecha de última actualización de cada fuente se consultan en el catálogo mediante la herramienta consultar_fuente_datos.   Este documento explica el procedimiento y no contiene el estado ni las fechas de ninguna fuente concreta.
""",
}
for nombre, contenido in fuentes.items():
    (carpeta / nombre).write_text(contenido, encoding="utf-8")
print("Archivos:", ", ".join(fuentes))

Archivos: tipos_de_informe.md, reglas_de_borrador.md


## 3. Lectura y limpieza
Conservamos archivo, título, versión y huella del original. Reducimos espacios y líneas repetidas; mantenemos párrafos, negaciones y condiciones.

In [24]:
documentos = []
for ruta in sorted(carpeta.glob("*.md")):
    original = ruta.read_text(encoding="utf-8")
    limpio = re.sub(r"[ \t]+", " ", original)
    limpio = re.sub(r"\n{3,}", "\n\n", limpio).strip()
    documentos.append({
        "document_id": ruta.stem, "source": ruta.name,
        "title": limpio.splitlines()[0].lstrip("# "), "version": "1.0-ficticia", "origin": ORIGEN,
        "source_hash": hashlib.sha256(original.encode("utf-8")).hexdigest(),
        "texto": limpio,
    })
    print(f"{ruta.name}: {len(original)} → {len(limpio)} caracteres")
print("\nMuestra del texto limpio:\n", documentos[0]["texto"][:450])

reglas_de_borrador.md: 1019 → 1016 caracteres
tipos_de_informe.md: 847 → 844 caracteres

Muestra del texto limpio:
 # Reglas de redacción de borradores

## Datos ausentes e incompletos

Cada métrica solicitada se contrasta con las métricas disponibles en la fuente antes de redactar. Una métrica que la fuente no ofrece se escribe como [DATO AUSENTE] en el lugar donde iría la cifra. El borrador no reemplaza ese vacío con una estimación ni con el valor de otro periodo. Si la fuente ofrece la métrica pero con registros faltantes, por ejemplo días sin datos dentro 


## 4. Dos estrategias de fragmentación
**Ventanas:** recorremos cada sección por tokens, con solapamiento. **Párrafos:** conservamos sus límites y subdividimos solo los párrafos que exceden el presupuesto. En la segunda estrategia no repetimos texto entre párrafos; sí entre ventanas de un párrafo largo.

Ambas incluyen título y sección en el texto vectorizado. Conservamos el mismo modelo, presupuesto y medida de búsqueda.

In [25]:
def ventanas(texto, limite):
    offsets = tokenizador(texto, add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"]
    if limite <= SOLAPAMIENTO:
        raise ValueError("El encabezado deja muy poco espacio para el contenido.")
    for inicio in range(0, len(offsets), limite - SOLAPAMIENTO):
        fin = min(inicio + limite, len(offsets))
        yield texto[offsets[inicio][0]:offsets[fin - 1][1]].strip()
        if fin == len(offsets):
            break

def fragmentar(estrategia):
    registros = []
    for documento in documentos:
        for numero, bloque in enumerate(re.split(r"(?m)^## ", documento["texto"])[1:], 1):
            seccion, cuerpo = bloque.split("\n", 1)
            encabezado = documento["title"] + " / " + seccion + "\n"
            limite = MAX_TOKENS - contar_tokens(encabezado) - 2
            partes = [cuerpo] if estrategia == "ventanas" else re.split(r"\n\s*\n", cuerpo.strip())
            for parte in partes:
                for texto in ventanas(parte.strip(), limite):
                    contenido = encabezado + texto
                    tokens = contar_tokens(contenido)
                    if tokens > min(MAX_TOKENS, modelo.max_seq_length):
                        raise ValueError("Un fragmento supera el límite: ajuste el presupuesto.")
                    registros.append({
                        **{k: documento[k] for k in ["document_id", "source", "title", "version", "origin", "source_hash"]},
                        "chunk_id": f"{documento['document_id']}:{documento['version']}:{estrategia}:{len(registros)}",
                        "section": seccion, "section_number": numero, "tokens": tokens, "texto": contenido,
                    })
    return registros

fragmentos = {nombre: fragmentar(nombre) for nombre in ["ventanas", "parrafos"]}
for nombre, registros in fragmentos.items():
    print(f"\n{nombre}: {len(registros)} fragmentos; máximo {max(r['tokens'] for r in registros)} tokens")
    for registro in registros[:2]:
        print(registro["chunk_id"], "|", registro["source"], "|", registro["tokens"], "tokens")
        print(registro["texto"], "\n")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (134 > 128). Running this sequence through the model will result in indexing errors



ventanas: 5 fragmentos; máximo 126 tokens
reglas_de_borrador:1.0-ficticia:ventanas:0 | reglas_de_borrador.md | 126 tokens
Reglas de redacción de borradores / Datos ausentes e incompletos
Cada métrica solicitada se contrasta con las métricas disponibles en la fuente antes de redactar. Una métrica que la fuente no ofrece se escribe como [DATO AUSENTE] en el lugar donde iría la cifra. El borrador no reemplaza ese vacío con una estimación ni con el valor de otro periodo. Si la fuente ofrece la métrica pero con registros faltantes, por ejemplo días sin datos dentro del periodo, la métrica se marca como [DATO INCOMPLETO 

reglas_de_borrador:1.0-ficticia:ventanas:1 | reglas_de_borrador.md | 56 tokens
Reglas de redacción de borradores / Datos ausentes e incompletos
métrica se marca como [DATO INCOMPLETO]. En ese caso la cifra puede aparecer, pero no se presenta como verificada y se explica qué parte del periodo falta. 


parrafos: 6 fragmentos; máximo 126 tokens
reglas_de_borrador:1.0-fictici

## 5. Consulta del usuario y comparación
Calculamos los embeddings de ambos conjuntos una vez. Después escribimos la pregunta en **CONSULTA** y comparamos los tres primeros resultados. Cada resultado conserva el archivo y la sección; vuelvan al original para comprobar su contenido. La similitud coseno ordena candidatos, no mide certeza.

In [26]:
matrices = {
    nombre: modelo.encode([r["texto"] for r in registros], normalize_embeddings=True, show_progress_bar=False)
    for nombre, registros in fragmentos.items()
}

def buscar(consulta, registros, matriz, k=3):
    if not consulta.strip():
        raise ValueError("Escriba una consulta.")
    if contar_tokens(consulta) > modelo.max_seq_length:
        raise ValueError("La consulta supera el límite del modelo.")
    vector = modelo.encode(consulta, normalize_embeddings=True)
    puntajes = matriz @ vector
    return [(registros[int(i)], float(puntajes[i])) for i in np.argsort(-puntajes)[:k]]

In [27]:
#@title Escriba aquí la consulta
CONSULTA = "¿Puedo presentar como verificada una métrica a la que le faltan días de datos?" #@param {type:"string"}

for nombre in fragmentos:
    print("\nESTRATEGIA:", nombre)
    for registro, puntaje in buscar(CONSULTA, fragmentos[nombre], matrices[nombre]):
        print(f"{registro['source']} · {registro['section']} · coseno {puntaje:.3f}")
        print(registro["chunk_id"])
        print(registro["texto"], "\n")


ESTRATEGIA: ventanas
reglas_de_borrador.md · Datos ausentes e incompletos · coseno 0.674
reglas_de_borrador:1.0-ficticia:ventanas:0
Reglas de redacción de borradores / Datos ausentes e incompletos
Cada métrica solicitada se contrasta con las métricas disponibles en la fuente antes de redactar. Una métrica que la fuente no ofrece se escribe como [DATO AUSENTE] en el lugar donde iría la cifra. El borrador no reemplaza ese vacío con una estimación ni con el valor de otro periodo. Si la fuente ofrece la métrica pero con registros faltantes, por ejemplo días sin datos dentro del periodo, la métrica se marca como [DATO INCOMPLETO 

reglas_de_borrador.md · Catálogo de fuentes · coseno 0.589
reglas_de_borrador:1.0-ficticia:ventanas:2
Reglas de redacción de borradores / Catálogo de fuentes
Las fuentes de datos se identifican con un código de la forma DS-0000, por ejemplo DS-1001. El periodo cubierto, las métricas disponibles y la fecha de última actualización de cada fuente se consultan en el 

In [28]:
PREGUNTAS = [
    "¿Puedo presentar como verificada una métrica a la que le faltan días de datos?",
    "Necesito el documento para la gerencia con las cifras más importantes",
    "¿Cuál es la última actualización de DS-1001?",
    "¿Cómo calculo la tasa de retención de clientes?",
]
for p in PREGUNTAS:
    print("\nPREGUNTA:", p)
    for nombre in fragmentos:
        print(" ", nombre)
        for registro, puntaje in buscar(p, fragmentos[nombre], matrices[nombre]):
            print(f"    {registro['chunk_id'].split(':')[-1]:>2} · {registro['source']} · {registro['section']} · {puntaje:.3f}")

# ¿El mejor resultado de la pregunta 1 trae la condición y su consecuencia?
for nombre in fragmentos:
    top = buscar(PREGUNTAS[0], fragmentos[nombre], matrices[nombre], k=1)[0][0]
    print(nombre, "| condición:", "DATO INCOMPLETO" in top["texto"], "| consecuencia:", "verificada" in top["texto"])


PREGUNTA: ¿Puedo presentar como verificada una métrica a la que le faltan días de datos?
  ventanas
     0 · reglas_de_borrador.md · Datos ausentes e incompletos · 0.674
     2 · reglas_de_borrador.md · Catálogo de fuentes · 0.589
     1 · reglas_de_borrador.md · Datos ausentes e incompletos · 0.589
  parrafos
     0 · reglas_de_borrador.md · Datos ausentes e incompletos · 0.674
     2 · reglas_de_borrador.md · Catálogo de fuentes · 0.589
     1 · reglas_de_borrador.md · Datos ausentes e incompletos · 0.589

PREGUNTA: Necesito el documento para la gerencia con las cifras más importantes
  ventanas
     3 · tipos_de_informe.md · Informe ejecutivo · 0.556
     4 · tipos_de_informe.md · Informe técnico · 0.363
     2 · reglas_de_borrador.md · Catálogo de fuentes · 0.314
  parrafos
     3 · tipos_de_informe.md · Informe ejecutivo · 0.580
     4 · tipos_de_informe.md · Informe ejecutivo · 0.408
     5 · tipos_de_informe.md · Informe técnico · 0.363

PREGUNTA: ¿Cuál es la última actualizaci

## 6. Guardar y volver a cargar el índice
Elijan una estrategia después de comparar. Guardamos vectores, registros y configuración; luego los leemos nuevamente y repetimos la consulta. Esta colección pequeña utiliza una matriz para búsqueda exacta. En Colab descarguen el ZIP para conservarlo fuera de la sesión.

In [29]:
ESTRATEGIA = "parrafos" #@param ["ventanas", "parrafos"]
destino = Path("indice_clase07")
destino.mkdir(exist_ok=True)
configuracion = {
    "modelo": MODELO, "revision": REVISION,
    "estrategia": ESTRATEGIA, "max_tokens": MAX_TOKENS, "solapamiento": SOLAPAMIENTO,
    "normalizados": True, "similitud": "coseno", "max_seq_length": modelo.max_seq_length,
}
np.save(destino / "vectores.npy", matrices[ESTRATEGIA])
(destino / "registros.json").write_text(json.dumps(fragmentos[ESTRATEGIA], ensure_ascii=False, indent=2), encoding="utf-8")
(destino / "configuracion.json").write_text(json.dumps(configuracion, indent=2), encoding="utf-8")

# Leemos desde disco: no recalculamos los embeddings del corpus.
registros_cargados = json.loads((destino / "registros.json").read_text(encoding="utf-8"))
vectores_cargados = np.load(destino / "vectores.npy", allow_pickle=False)
config_cargada = json.loads((destino / "configuracion.json").read_text(encoding="utf-8"))
assert config_cargada["modelo"] == MODELO and vectores_cargados.shape[0] == len(registros_cargados)
antes = buscar(CONSULTA, fragmentos[ESTRATEGIA], matrices[ESTRATEGIA])
despues = buscar(CONSULTA, registros_cargados, vectores_cargados)
assert [r["chunk_id"] for r, _ in antes] == [r["chunk_id"] for r, _ in despues]
assert np.allclose([p for _, p in antes], [p for _, p in despues])
print("Índice cargado: mismos fragmentos y puntajes para la consulta.")

with zipfile.ZipFile("Clase_07_Indice_y_Corpus.zip", "w", zipfile.ZIP_DEFLATED) as archivo:
    for ruta in [*destino.iterdir(), *carpeta.glob("*.md")]:
        archivo.write(ruta, arcname=ruta.as_posix())
print("Archivo para conservar:", "Clase_07_Indice_y_Corpus.zip")

Índice cargado: mismos fragmentos y puntajes para la consulta.
Archivo para conservar: Clase_07_Indice_y_Corpus.zip


Para usar este índice en otra sesión: descompriman el ZIP, lean la configuración y carguen `SentenceTransformer(config_cargada["modelo"], revision=config_cargada["revision"])`. Lean los registros y la matriz como en la celda anterior. Las consultas deben usar esa misma representación.

In [30]:
#@title Descargar el índice y los documentos (opcional)
DESCARGAR = False #@param {type:"boolean"}
if DESCARGAR:
    from google.colab import files
    files.download("Clase_07_Indice_y_Corpus.zip")

## 7. Trabajo del equipo

Reemplazamos los dos documentos de la demo por fuentes del proyecto (`tipos_de_informe.md` y `reglas_de_borrador.md`) con título, versión `1.0-ficticia` y origen registrados. Comparamos las dos estrategias con tres preguntas además de la consulta inicial y guardamos el índice elegido.

| Pregunta | Evidencia esperada | Ventanas | Párrafos |
|---|---|---|---|
| ¿Puedo presentar como verificada una métrica a la que le faltan días de datos? | reglas_de_borrador · Datos ausentes e incompletos | Sí: fragmento 0 (0.674) trae la condición; la consecuencia está en el 1 (0.589), empatado con el de Catálogo (0.589) | Igual, mismos fragmentos y puntajes |
| Necesito el documento para la gerencia con las cifras más importantes | tipos_de_informe · Informe ejecutivo | Sí: fragmento 3 (0.556) | Sí: fragmento 3 (0.580); el párrafo de "cifras parciales" queda aparte (fragmento 4, 0.408) |
| ¿Cuál es la última actualización de DS-1001? | reglas_de_borrador · Catálogo de fuentes (explica dónde consultar, no trae la fecha) | Sí: fragmento 2 (0.550); no trae la fecha | Igual (0.550) |
| ¿Cómo calculo la tasa de retención de clientes? | Ninguno: fuera de cobertura | Devuelve Informe técnico (0.329) aunque no responde | Igual (0.329) |

* **Resultado útil:** "gerencia" llegó a la sección ejecutiva, que dice "dirección" (0.556 y 0.580). Para DS-1001 la búsqueda ubicó el procedimiento del catálogo (0.550), que no trae la fecha. El código sí aparece literalmente en ese pasaje ("por ejemplo DS-1001"), así que aquí no demuestra búsqueda semántica.

* **Fallo:** en la pregunta 1 la condición y la consecuencia quedaron en fragmentos distintos en ambas estrategias. El fragmento 0 termina en `[DATO INCOMPLETO ` sin cerrar el corchete, y la consecuencia ("no se presenta como verificada") está en el 1. Ese fragmento empató con el de Catálogo, que no responde la pregunta, así que con K=2 puede quedar fuera. El solapamiento de 12 tokens repite el inicio de `[DATO INCOMPLETO]` en el fragmento 1 y ayuda a reconocerlo.

* **Estrategia elegida y por qué:** párrafos. Con 128 tokens la diferencia aparece solo en `tipos_de_informe`: la sección ejecutiva queda en dos fragmentos y la regla del periodo no cerrado se recupera sola (0.408). El puntaje de la pregunta de la gerencia fue un poco mayor (0.580 vs 0.556). En `reglas_de_borrador` ambas estrategias dan lo mismo. Con cuatro preguntas la diferencia es pequeña y no la damos por demostrada.

**Antes de terminar:** ¿cada fragmento permite volver a la fuente?, ¿se conservan las condiciones del procedimiento?, ¿el índice cargado devuelve los mismos resultados?

Documentación: [modelo de embeddings](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2) · [Sentence Transformers](https://sbert.net/docs/package_reference/sentence_transformer/model.html).